In [10]:
!uname -a && cat /etc/*release
!pwd
!ls -la

Linux a8d579438f76 6.6.113+ #1 SMP Mon Feb  2 12:27:57 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux
DISTRIB_ID=Ubuntu
DISTRIB_RELEASE=22.04
DISTRIB_CODENAME=jammy
DISTRIB_DESCRIPTION="Ubuntu 22.04.5 LTS"
PRETTY_NAME="Ubuntu 22.04.5 LTS"
NAME="Ubuntu"
VERSION_ID="22.04"
VERSION="22.04.5 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=jammy
/content
total 1016
drwxr-xr-x 1 root root    4096 Mar 17 09:07 .
drwxr-xr-x 1 root root    4096 Mar 17 08:48 ..
drwxr-xr-x 4 root root    4096 Feb  6 14:31 .config
-rwxr-xr-x 1 root root 1011768 Mar 17 09:06 cuda_matrixmult
-rw-r--r-- 1 root root    3478 Mar 17 09:06 cuda_matrixmult.cu
-rw-r--r-- 1 root root     516 Mar 17 09:07 mat-res.txt
drwxr-xr-x 1 root root    4096 Feb  6 14:31 sample_data


In [11]:
!nvidia-smi

Tue Mar 17 09:08:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [57]:
cell_str='''
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>

#define TILE_SIZE 32

// Kernel CUDA super-ottimizzato: Tiling in Shared Memory + Singola Precisione (FP32)
__global__ void matMulTiledFloat(const float *A, const float *B, float *C, int n) {
    // 1. Identifichiamo dove siamo nella matrice globale
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    // 2. Alloco la Shared Memory per il blocco (ora usiamo i float!)
    __shared__ float tileA[TILE_SIZE][TILE_SIZE];
    __shared__ float tileB[TILE_SIZE][TILE_SIZE];

    float sum = 0.0f;

    // 3. Facciamo scorrere la piastrella lungo la direzione da calcolare
    int numTiles = (n + TILE_SIZE - 1) / TILE_SIZE;

    for (int t = 0; t < numTiles; ++t) {
        // I thread caricano i dati dalla memoria Globale a quella Shared
        if (row < n && (t * TILE_SIZE + threadIdx.x) < n)
            tileA[threadIdx.y][threadIdx.x] = A[row * n + t * TILE_SIZE + threadIdx.x];
        else
            tileA[threadIdx.y][threadIdx.x] = 0.0f;

        if (col < n && (t * TILE_SIZE + threadIdx.y) < n)
            tileB[threadIdx.y][threadIdx.x] = B[(t * TILE_SIZE + threadIdx.y) * n + col];
        else
            tileB[threadIdx.y][threadIdx.x] = 0.0f;

        // Barriera di sincronizzazione: aspettiamo il caricamento completo
        __syncthreads();

        // 4. Moltiplicazione a manetta sui dati in cache
        for (int k = 0; k < TILE_SIZE; ++k) {
            sum += tileA[threadIdx.y][k] * tileB[k][threadIdx.x];
        }

        // Seconda barriera: aspettiamo che tutti abbiano finito prima del prossimo giro
        __syncthreads();
    }

    // 5. Scrittura del risultato finale nella memoria globale
    if (row < n && col < n) {
        C[row * n + col] = sum;
    }
}

int main(int argc, char **argv) {
    if (argc < 2) {
        fprintf(stderr, "Uso: %s <n>\\n", argv[0]);
        return 1;
    }

    int n = atoi(argv[1]);
    // Calcoliamo i byte basandoci su float (4 byte) anziché double (8 byte)
    size_t bytes = n * n * sizeof(float);

    float *h_a = (float*)malloc(bytes);
    float *h_b = (float*)malloc(bytes);
    float *h_c = (float*)malloc(bytes);

    // Inizializzazione con letterali float (il suffisso 'f' è importante!)
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            h_a[i * n + j] = 2.0f;
            h_b[i * n + j] = 3.0f;
            h_c[i * n + j] = 0.0f;
        }
    }

    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(TILE_SIZE, TILE_SIZE);
    dim3 numBlocks((n + TILE_SIZE - 1) / TILE_SIZE, (n + TILE_SIZE - 1) / TILE_SIZE);

    clock_t start = clock();

    // Lanciamo il kernel in singola precisione!
    matMulTiledFloat<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, n);

    cudaDeviceSynchronize();
    clock_t end = clock();

    printf("Execution Time (Tiled FP32): %.4f seconds\\n", (double)(end - start) / CLOCKS_PER_SEC);

    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    // Salviamo un piccolo sample per verificare che i conti tornino
    FILE *f = fopen("mat-res.txt", "w");
    if (f) {
        fprintf(f, "%d\\n\\n", n);
        int sample = (n < 100) ? n : 10;
        for (int i = 0; i < sample; i++) {
            for (int j = 0; j < sample; j++) {
                fprintf(f, "%.0f ", h_c[i * n + j]);
            }
            fprintf(f, "\\n");
        }
        fclose(f);
    }

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);

    return 0;
}
'''
with open('cuda_matrixmult.cu', 'w') as f:
    f.write(cell_str)

In [58]:
!nvcc -arch=sm_75 cuda_matrixmult.cu -o cuda_matrixmult

In [59]:
!nvprof ./cuda_matrixmult 15000

==14085== NVPROF is profiling process 14085, command: ./cuda_matrixmult 15000
Execution Time (Tiled FP32): 7.1487 seconds
==14085== Profiling application: ./cuda_matrixmult 15000
==14085== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   92.11%  7.15625s         1  7.15625s  7.15625s  7.15625s  matMulTiledFloat(float const *, float const *, float*, int)
                    5.12%  397.90ms         2  198.95ms  197.38ms  200.52ms  [CUDA memcpy HtoD]
                    2.77%  215.49ms         1  215.49ms  215.49ms  215.49ms  [CUDA memcpy DtoH]
      API calls:   89.94%  7.15627s         1  7.15627s  7.15627s  7.15627s  cudaDeviceSynchronize
                    7.72%  614.27ms         3  204.76ms  197.68ms  215.84ms  cudaMemcpy
                    2.22%  176.96ms         3  58.987ms  135.80us  176.61ms  cudaMalloc
                    0.07%  5.9666ms         3  1.9889ms  946.34us  2.8951ms  cudaFree
                    0